# Laboratorio 4: Análisis geoespacial de cianobacterias
## Avance - ejercicios 1 a 4

**Objetivo.** Analizar la señal de cianobacteria en los lagos de Amatitlán y Atitlán a partir de las 22 fechas oficiales Sentinel-2, con una cadena reproducible desde la descarga hasta el análisis temporal inicial.

**Fuentes.** Copernicus Data Space Ecosystem (Sentinel-2) y el script CyanoLakes Chlorophyll-a NDCI L1C de Sentinel Hub. Los ejercicios 5 a 8 se dejan explícitamente pendientes para la fase 2.

### 0. Dependencias

Las librerías utilizadas en este análisis están listadas en `requirements.txt`. La autenticación con Copernicus Data Space se realiza mediante OIDC y nunca se versionan credenciales. Abra Jupyter desde la raíz del proyecto para que las rutas relativas funcionen.

### Repositorio del proyecto

Código, historial de contribuciones e instrucciones de reproducción: [DS-LAB04 en GitHub](https://github.com/jcdiegolopez/DS-LAB04).

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Abra este notebook desde la carpeta raíz del proyecto DS-LAB04.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from configuracion import LAKES, OFFICIAL_SCENES, REQUIRED_BANDS
from descarga import (
    build_inventory, connect_copernicus, download_official_scene,
    save_inventory, update_download_status, validate_geotiff,
)
from indices import CYANO_SCRIPT_NAME, CYANO_SCRIPT_URL, build_metrics_table, metrics_for_scene
from analisis_temporal import detect_peaks, plot_cyano_timeseries, temporal_interpretation
from control_calidad import build_quality_checklist, save_quality_checklist

print(f'Bandas mínimas solicitadas: {REQUIRED_BANDS}')
print(f'Lagos configurados: {[item["display_name"] for item in LAKES.values()]}')

## 1. Conexión con la API Sentinel-2 (openEO)

La conexión a Copernicus Data Space permite consultar la colección Sentinel-2 L2A y generar las descargas necesarias para el análisis.

In [ ]:
connection = connect_copernicus()
print('Conexión autenticada:', connection)

## 2. Obtención y trazabilidad de datos raster

El laboratorio exige trabajar únicamente con 11 fechas por lago. La validación siguiente debe dar exactamente 22 filas, 11 para Amatitlán y 11 para Atitlán.

In [ ]:
inventory = build_inventory()
assert len(inventory) == 22, 'Deben existir 22 escenas oficiales en total.'
assert inventory.groupby('lago').size().to_dict() == {'amatitlan': 11, 'atitlan': 11}
inventory_path = save_inventory(inventory)
display(inventory)
print(f'Inventario guardado en: {inventory_path}')

In [ ]:
# Control de trazabilidad: fechas, satélite y nubosidad reportados en la guía.
display(inventory.groupby('lago', as_index=False).agg(
    imagenes=('fecha', 'count'),
    primera_fecha=('fecha', 'min'),
    ultima_fecha=('fecha', 'max'),
    nubosidad_promedio_reportada=('nubosidad_oficial_pct', 'mean'),
))

### 2.1 Descarga mínima de una escena

La descarga se limita a B03, B04 y B08. Se valida una imagen de cada lago antes de procesar el conjunto completo. La fecha 2026-02-07 de Amatitlán cuenta con cobertura válida parcial (aprox. 57.1%), observación que se mantiene en el inventario.

In [ ]:
atitlan_test = download_official_scene(connection, 'atitlan', '2025-01-18')
amatitlan_test = download_official_scene(connection, 'amatitlan', '2025-01-28')
display(pd.DataFrame([validate_geotiff(atitlan_test), validate_geotiff(amatitlan_test)]))

In [ ]:

for row in inventory.itertuples(index=False):
    path = download_official_scene(connection, row.lago, row.fecha)
    print(f'Descargado: {path}')

### 2.2 Control de calidad de las descargas

El inventario registra el estado `descargado_validado` cuando el archivo es georreferenciado y contiene las tres bandas solicitadas.

In [ ]:
inventory_final = update_download_status(inventory)
save_inventory(inventory_final)
display(inventory_final[['lago', 'fecha', 'satelite', 'bandas_solicitadas', 'estado_descarga', 'observaciones']])

pendientes = inventory_final.query("estado_descarga != 'descargado_validado'")
print(f'Escenas validadas: {(inventory_final.estado_descarga == "descargado_validado").sum()} de {len(inventory_final)}')
if not pendientes.empty:
    print('Pendientes o con incidencias:')
    display(pendientes[['lago', 'fecha', 'estado_descarga', 'observaciones']])

lista_calidad = build_quality_checklist(inventory_final)
ruta_lista_calidad = save_quality_checklist(lista_calidad)
display(lista_calidad)
print(f'Lista de control guardada en: {ruta_lista_calidad}')

## 3. Índices espectrales

Se calculan NDVI = (B08 − B04) / (B08 + B04) y NDWI = (B03 − B08) / (B03 + B08). Las divisiones por cero, nodata y píxeles fuera de una máscara del lago se convierten en `NaN`.

La señal de cianobacteria se estima con **CyanoLakes Chlorophyll-a NDCI L1C (Kravitz & Matthews, 2020)**, documentado en el [Cyano Detection Script de Sentinel Hub](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/cyanobacteria_chla_ndci_l1c/). El visualizador por defecto del script produce colores RGB; para promedios, mapas y picos se necesita además una exportación **numérica** que conserve la fórmula, las máscaras y la grilla. El producto se guarda como `data/cyano/<lago>/<lago>_<fecha>_cyano.tif` y debe tener una banda alineada al raster mínimo.

Los cálculos convierten nodata, divisiones inválidas y píxeles fuera de la máscara en `NaN`. Mientras no se incorpore el GeoJSON de cada lago, la máscara solo excluye nodata; los resultados deben considerarse preliminares.

In [ ]:
insumos_listos = (lista_calidad['estado'] == 'cumple').all()
if insumos_listos:
    metricas = build_metrics_table(inventory_final)
    display(metricas)
    print('Tabla guardada en outputs/tablas/metricas_por_fecha.csv')
    print('Incidencias guardadas en outputs/tablas/incidencias_persona2.csv')
else:
    metricas = None
    print('No se calcularon métricas: complete los elementos pendientes de la lista de control antes de interpretar resultados.')

In [ ]:
if metricas is not None:
    vmin = metricas['cianobacteria_promedio'].min()
    vmax = metricas['cianobacteria_promedio'].max()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    for ax, (lago, grupo) in zip(axes, metricas.groupby('lago')):
        escena = grupo.loc[grupo['cianobacteria_promedio'].idxmax()]
        _, capas = metrics_for_scene(lago, escena['fecha'])
        imagen = capas['cianobacteria'].copy()
        imagen[~capas['valid_mask']] = np.nan
        mapa = ax.imshow(imagen, cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set(title=f"{LAKES[lago]['display_name']} - {escena['fecha']}", xticks=[], yticks=[])
    fig.colorbar(mapa, ax=axes, label='Indicador numérico de cianobacteria')
    fig.suptitle('Cianobacteria: fecha de mayor promedio por lago (escala común)')
    plt.show()
else:
    print('Mapa pendiente: aún no hay métricas verificables.')

## 4. Análisis temporal

Se usa como fecha crítica un **máximo local** cuyo promedio de cianobacteria es igual o superior al **percentil 75 de su propio lago**. El criterio se calcula por separado para no confundir las escalas y condiciones de Amatitlán y Atitlán.

In [ ]:
if metricas is not None:
    serie_temporal = detect_peaks(metricas)
    display(serie_temporal[['lago', 'fecha', 'pixeles_validos', 'cobertura_valida_pct', 'cianobacteria_promedio', 'umbral_percentil_75', 'es_pico', 'incidencia']])

    ax = plot_cyano_timeseries(serie_temporal)
    plt.show()

    resumen_temporal = temporal_interpretation(serie_temporal)
    display(resumen_temporal)
else:
    serie_temporal = None
    print('Análisis temporal pendiente: aún no hay métricas verificables.')

### Interpretación temporal inicial

La gráfica se lee de izquierda a derecha: cada punto es el promedio de píxeles válidos de una fecha oficial y las estrellas identifican fechas críticas. La tabla `resumen_temporal` reporta el máximo observado y el número de picos por lago. Esos son **hechos extraídos de los datos**. Lluvia, viento, temperatura, aportes de nutrientes y mezcla del agua son hipótesis que podrían explicar cambios, pero solo deben afirmarse tras contrastarlas con datos auxiliares; además, nubes, cobertura parcial o ausencia de máscara pueden alterar un promedio.

## Metodología y limitaciones del avance

Se usan imágenes Sentinel-2 en las 11 fechas oficiales por lago. B04 y B08 permiten calcular NDVI; B03 y B08, NDWI. La estimación de cianobacteria se deriva del producto numérico documentado del algoritmo CyanoLakes, no de una interpretación visual de colores. Las métricas se calculan únicamente sobre píxeles válidos y la misma regla de máscara se aplica a todas las fechas.

El archivo `outputs/tablas/inventario_datos.csv` conserva la trazabilidad de cada imagen y `outputs/tablas/lista_control_calidad_avance.csv` registra si los insumos necesarios están disponibles. La ausencia de una máscara vectorial precisa, nubes, cobertura parcial y la naturaleza indirecta de la estimación limitan las conclusiones.

## Conclusiones del avance

El avance deja una cadena reproducible para validar las fechas, descargar las bandas mínimas, calcular índices y detectar picos con un criterio explícito. Las conclusiones ambientales se incorporarán únicamente después de que las figuras y la tabla de métricas se generen con los 22 pares de insumos validados.

## 5. Análisis espacial - pendiente de fase 2

Se incorporarán mapas comparativos, extensión de valores altos y persistencia espacial.

## 6. Correlaciones - pendiente de fase 2

Se evaluarán las relaciones entre cianobacteria, NDVI y NDWI.

## 7. Comparación de lagos - pendiente de fase 2

Se compararán intensidad, frecuencia y causas potenciales con evidencia de tablas y figuras.

## 8. Análisis exploratorio adicional - pendiente de fase 2

Se analizarán área alta, persistencia, distribuciones y estacionalidad.